In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

import time

project_root = Path(
    r"C:\Users\rajur\OneDrive\Desktop\next-move-ai"
)

feature_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_4_question_difficulty"
    / "base_features"
)

prediction_dir = (
    project_root
    / "data"
    / "processed"
    / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Feature directory:", feature_dir)
print("Prediction directory:", prediction_dir)

Project root: C:\Users\rajur\OneDrive\Desktop\next-move-ai
Feature directory: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_4_question_difficulty\base_features
Prediction directory: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions


In [2]:
fold_number = 1

train_path = feature_dir / f"fold_{fold_number}_train.csv"
val_path = feature_dir / f"fold_{fold_number}_validation.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

feature_columns = [
    "question_smoothed_accuracy"
]

target_column = "correct"

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_val = val_df[feature_columns]
y_val = val_df[target_column]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

print("\nTraining feature:")
print(X_train.describe())

print("\nTraining target distribution:")
print(y_train.value_counts())

Training: (177324, 1) (177324,)
Validation: (42659, 1) (42659,)

Training feature:
       question_smoothed_accuracy
count               177324.000000
mean                     0.663163
std                      0.097213
min                      0.167892
25%                      0.607347
50%                      0.676779
75%                      0.731524
max                      1.000000

Training target distribution:
correct
1    117058
0     60266
Name: count, dtype: int64


In [3]:
fold_results = []
all_predictions = []

feature_columns = [
    "question_smoothed_accuracy"
]

target_column = "correct"

for fold_number in range(1, 6):

    print(f"\n{'='*45}")
    print(f"Training V4.1 — Fold {fold_number}")
    print(f"{'='*45}")

    # Load fold-specific datasets
    train_df = pd.read_csv(
        feature_dir / f"fold_{fold_number}_train.csv"
    )

    val_df = pd.read_csv(
        feature_dir / f"fold_{fold_number}_validation.csv"
    )

    X_train = train_df[feature_columns]
    y_train = train_df[target_column]

    X_val = val_df[feature_columns]
    y_val = val_df[target_column]

    # Build a fresh model for this fold
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        ))
    ])

    # Train
    start_time = time.perf_counter()

    model.fit(X_train, y_train)

    training_time = time.perf_counter() - start_time

    # Predict probabilities
    start_time = time.perf_counter()

    y_pred_proba = model.predict_proba(X_val)[:, 1]

    prediction_time = time.perf_counter() - start_time

    # Evaluate
    fold_log_loss = log_loss(y_val, y_pred_proba)
    fold_brier = brier_score_loss(y_val, y_pred_proba)

    # Extract standardized coefficient
    logistic_model = model.named_steps["logistic_regression"]
    coefficient = logistic_model.coef_[0][0]
    intercept = logistic_model.intercept_[0]

    fold_results.append({
        "fold": fold_number,
        "log_loss": fold_log_loss,
        "brier_score": fold_brier,
        "coefficient": coefficient,
        "intercept": intercept,
        "training_time": training_time,
        "prediction_time": prediction_time
    })

    # Store OOF predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_df["user_id"].to_numpy(),
        "order_id": val_df["order_id"].to_numpy(),
        "fold": fold_number,
        "y_true": y_val.to_numpy(),
        "y_pred_proba": y_pred_proba,
        "model": "v4.1"
    })

    all_predictions.append(fold_predictions)

    print(f"Training interactions: {len(train_df)}")
    print(f"Validation interactions: {len(val_df)}")
    print(f"Log Loss: {fold_log_loss:.6f}")
    print(f"Brier Score: {fold_brier:.6f}")
    print(f"Coefficient: {coefficient:.6f}")
    print(f"Intercept: {intercept:.6f}")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Prediction time: {prediction_time:.2f} seconds")


Training V4.1 — Fold 1
Training interactions: 177324
Validation interactions: 42659
Log Loss: 0.618620
Brier Score: 0.214413
Coefficient: 0.485677
Intercept: 0.694693
Training time: 0.12 seconds
Prediction time: 0.00 seconds

Training V4.1 — Fold 2
Training interactions: 176273
Validation interactions: 43710
Log Loss: 0.607914
Brier Score: 0.209324
Coefficient: 0.489523
Intercept: 0.668927
Training time: 0.05 seconds
Prediction time: 0.00 seconds

Training V4.1 — Fold 3
Training interactions: 170186
Validation interactions: 49797
Log Loss: 0.625960
Brier Score: 0.217814
Coefficient: 0.481382
Intercept: 0.710151
Training time: 0.04 seconds
Prediction time: 0.00 seconds

Training V4.1 — Fold 4
Training interactions: 178311
Validation interactions: 41672
Log Loss: 0.613760
Brier Score: 0.212157
Coefficient: 0.491339
Intercept: 0.683320
Training time: 0.04 seconds
Prediction time: 0.00 seconds

Training V4.1 — Fold 5
Training interactions: 177838
Validation interactions: 42145
Log Loss: 0

In [4]:
fold_results_df = pd.DataFrame(fold_results)

print("\nFOLD-WISE RESULTS")
print(fold_results_df.round(6).to_string(index=False))

print("\nAVERAGE CROSS-VALIDATION RESULTS")

print(
    "Average Log Loss:",
    fold_results_df["log_loss"].mean()
)

print(
    "Average Brier Score:",
    fold_results_df["brier_score"].mean()
)

print(
    "Average Coefficient:",
    fold_results_df["coefficient"].mean()
)


FOLD-WISE RESULTS
 fold  log_loss  brier_score  coefficient  intercept  training_time  prediction_time
    1  0.618620     0.214413     0.485677   0.694693       0.118181         0.002654
    2  0.607914     0.209324     0.489523   0.668927       0.052006         0.003685
    3  0.625960     0.217814     0.481382   0.710151       0.038050         0.002341
    4  0.613760     0.212157     0.491339   0.683320       0.041167         0.002570
    5  0.608712     0.209799     0.482255   0.682412       0.040644         0.002338

AVERAGE CROSS-VALIDATION RESULTS
Average Log Loss: 0.6149931596424467
Average Brier Score: 0.2127014296588125
Average Coefficient: 0.48603514930508124


In [5]:
oof_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

prediction_path = prediction_dir / "v4_1_predictions.csv"

oof_predictions.to_csv(
    prediction_path,
    index=False
)

print("OOF prediction shape:", oof_predictions.shape)
print("Saved to:", prediction_path)
print("Missing values:", oof_predictions.isna().sum().sum())
print(
    "Duplicate interactions:",
    oof_predictions.duplicated(["user_id", "order_id"]).sum()
)

OOF prediction shape: (219983, 6)
Saved to: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v4_1_predictions.csv
Missing values: 0
Duplicate interactions: 0


V4.1 — Question Smoothed Accuracy

Features:
1. question_smoothed_accuracy

Average CV Log Loss: 0.614993
Average CV Brier Score: 0.212701

Average Coefficient: 0.486035

Model:
StandardScaler + LogisticRegression

C = 1.0
max_iter = 1000
random_state = 42